In [ ]:
%load_ext autoreload
%autoreload 2

# Collect Data

In [ ]:
from docent_core._loader.load_inspect import load_inspect_eval

LOG_DIR_PREFIX = "/home/ubuntu/inspect_logs"
PICOCTF_TRANSCRIPTS = load_inspect_eval({"default_scaffold": f"{LOG_DIR_PREFIX}/intercode-4o-default-scaffold.eval",})
# AGENTHARM_TRANSCRIPTS = load_inspect_eval({"agentharm_sonnet35": f"{LOG_DIR_PREFIX}/agent-harm.eval",})
len(PICOCTF_TRANSCRIPTS)

In [ ]:
# dump the transcripts into a pickle
import pickle

from docent.data_models.metadata import BaseAgentRunMetadata
from pydantic import Field

class BWAgentRunMetadata(BaseAgentRunMetadata):
    question: str


BRIDGEWATER_TRANSCRIPTS = pickle.load(open(f'{LOG_DIR_PREFIX}/bridgewater_transcripts.pkl', 'rb'))

len(BRIDGEWATER_TRANSCRIPTS)

In [ ]:
from docent.sdk.client import Docent


docent = Docent(
    server_url="http://localhost:8889",
    web_url="http://localhost:3001",
    api_key="dk_6NMO8u7xE-i_enAAEJxeesDzkJyKdnPyBMrwSxxDoM8",
)
docent.add_agent_runs(
    collection_id="a9d81c4a-013f-4595-a709-a9b5bd8f39c5",
    agent_runs=BRIDGEWATER_TRANSCRIPTS,
)


In [ ]:
from docent.data_models.metadata import BaseAgentRunMetadata
from pydantic import Field


class CursorAgentRunMetadata(BaseAgentRunMetadata):
    general_explanation: str = Field(
        description="An explanation of the overall rollout structure"
    )
    task_id: str | None = Field(
        description="Prompt/input into the model", default=None
    )
    gold_diff: str = Field(
        description="The reference diff that the model is being judged against"
    )
    judge_prompt: str = Field(
        description="The prompt that was used to judge the model"
    )
    rollout_diff: str = Field(
        description="The diff generated by the model in the rollout"
    )
    llm_judge_data: str = Field(
        description="The response from the judge model"
    )
    rank: int = Field(
        description="The data-parallel rank of the model"
    )
    step: int = Field(
        description="The training step number"
    )
    rollout: int = Field(
        description="The rollout id of the training step"
    )

CURSOR_TRANSCRIPTS = pickle.load(open(f'/home/ubuntu/docent/cursor_new_2.pkl', 'rb'))
len(CURSOR_TRANSCRIPTS)

In [ ]:
class ElicitationMetadata(BaseAgentRunMetadata):
    run_id: str
    step: int
    rollout: int
    run_info: str

ELICITATION_TRANSCRIPTS = pickle.load(open(f'{LOG_DIR_PREFIX}/elicitation_transcripts.pkl', 'rb'))
len(ELICITATION_TRANSCRIPTS)

In [ ]:
from docent_core._loader.load_inspect import load_swebench

TRANSCRIPTS = load_swebench()

In [ ]:
from docent.sdk.client import Docent

docent = Docent(
    server_url="http://localhost:8889",
    web_url="http://localhost:3001",
    api_key="dk_8SjA-HAIwIfYBJiTvIcMYepBPvwxfRexBOoGIXKjHeQ",
)

docent.add_agent_runs(
    collection_id="c0e02882-24bb-450a-8711-c476543abcec",
    agent_runs=CURSOR_TRANSCRIPTS[:300],
)


# Basic loop

In [ ]:
from typing import cast
from docent.data_models.agent_run import AgentRun

from docent_core._ai_tools.clustering.cluster_assigner import LlmApiClusterAssigner
from docent_core._ai_tools.search import execute_search_2, execute_search_3
from docent_core._ai_tools.clustering.cluster_generator import propose_clusters

import random


class SearchRefiner:
    def __init__(self, ars: list[AgentRun], search_query: str):
        self.ars = ars[:300]
        self.search_query = search_query
        self.NUM_SEARCHES_MAX = 100 # max amount to cluster over

        self.search_results: list[str] = []
        self.near_search_results: list[str] = []

        self.positive_clusters: list[str] = []
        self.negative_clusters: list[str] = []

        self.positive_clusters_assignments: dict[str, list[int]] = {}
        self.negative_clusters_assignments: dict[str, list[int]] = {}

        self.total_search_results = 0
        self.total_neg_search_results = 0

        self.assigner = LlmApiClusterAssigner.from_o3_mini()


    async def get_initial_search_results(self):
        results = await execute_search_2(self.ars, self.search_query)
        for result in results:
            if result is None:
                continue
            self.search_results.extend(result)
        indices = [i for i in range(len(results)) if results[i] is not None]
        new_results = await execute_search_3(
            [self.ars[i] for i in indices],
            self.search_query,
            previous_results=[r for r in results if r is not None],
        )
        for result in new_results:
            if result is None:
                continue
            self.near_search_results.extend(result)
        
        
        random.seed(42)
        random.shuffle(self.search_results)
        random.seed(42)
        random.shuffle(self.near_search_results)
        self.total_search_results = len(self.search_results)
        self.total_neg_search_results = len(self.near_search_results)
        self.search_results = self.search_results[:self.NUM_SEARCHES_MAX]
        self.near_search_results = self.near_search_results[:self.NUM_SEARCHES_MAX]
        

    async def get_initial_clusters(self):
        self.positive_clusters = await propose_clusters(self.search_results, extra_instructions_list=[f"focus specifically on {self.search_query.split(".")[0]}"])
        self.negative_clusters = await propose_clusters(self.near_search_results, extra_instructions_list=[f"focus specifically on {self.search_query.split(".")[0]}"])


    async def get_assign_residuals(self):
        async def _assign(searches: list[str], clusters: list[str]) -> dict[str, list[int]]:
            full_searches: list[str] = []
            full_clusters: list[str] = []
            for c in clusters:
                full_searches.extend(searches)
                full_clusters.extend([c] * len(searches))
            assignments = await self.assigner.assign(full_searches, full_clusters)
            final_results: dict[str, list[int]] = {}
            for i, c in enumerate(clusters):
                subsequence = assignments[i * len(searches):(i + 1) * len(searches)]
                final_results[c] = [i for i, a in enumerate(subsequence) if a is not None and a[0]]
            return final_results

        self.positive_clusters_assignments = await _assign(self.search_results, self.positive_clusters)
        self.negative_clusters_assignments = await _assign(self.near_search_results, self.negative_clusters)

    async def visualize_search_results_for_cluster(self, pos_cluster: str, neg_cluster: str, max_examples: int = 10):
        initial_results = await execute_search_2(self.ars, self.search_query)
        indices = [i for i in range(len(initial_results)) if initial_results[i] is not None]
        pos_results = [cast(list[str], initial_results[i]) for i in indices]
        neg_results = await execute_search_3(
            [self.ars[i] for i in indices],
            self.search_query,
            previous_results=pos_results,
        )

        flat_to_pos_map: list[int] = []
        flat_to_neg_map: list[int] = []
        for i, pos_result in enumerate(pos_results):
            for _ in pos_result:
                flat_to_pos_map.append(i)
        for i, neg_result in enumerate(neg_results):
            if neg_result is None:
                continue
            for _ in neg_result:
                flat_to_neg_map.append(i)
        
        # print(pos_results)
        # print(flat_to_pos_map)

        flattened_pos_results = [item for sublist in pos_results for item in sublist]

        # print(flattened_pos_results)
        flattened_neg_results = [item for sublist in neg_results if sublist is not None for item in sublist]

        flattened_pos_indices = list(range(len(flattened_pos_results)))
        flattened_neg_indices = list(range(len(flattened_neg_results)))

        if pos_cluster != "":
            flattened_pos_indices = [i for i, res in enumerate(await self.assigner.assign(flattened_pos_results, [pos_cluster,] * len(flattened_pos_results))) if res is not None and res[0]]
        if neg_cluster != "":
            flattened_neg_indices = [i for i, res in enumerate(await self.assigner.assign(flattened_neg_results, [neg_cluster,] * len(flattened_neg_results))) if res is not None and res[0]]

        # print((await self.assigner.assign(flattened_pos_results, [pos_cluster,] * len(flattened_pos_results))))

        print(flattened_pos_indices)

        for i in range(min(max_examples, len(neg_results))):
            pos_indices = [elem for _, elem in enumerate(flattened_pos_indices) if flat_to_pos_map[elem] == i]
            # print(pos_indices)
            neg_indices = [elem for _, elem in enumerate(flattened_neg_indices) if flat_to_neg_map[elem] == i]
            if len(pos_indices) == 0 and len(neg_indices) == 0:
                continue
            print(f"Example {i}: Pos ({len(pos_indices)}), Neg ({len(neg_indices)})" + "=" * 50)
            for p in pos_indices:
                print(f"  {flattened_pos_results[p]}")
            print('-' * 50)
            for n in neg_indices:
                print(f"  {flattened_neg_results[n]}")

    def visualize_clusters(self, max_items_per_cluster: int = 10):
        def _visualize(clusters: list[str], searches: list[str], assignments: dict[str, list[int]]):
            for c in clusters:
                print("-" * 100)
                print(f"Cluster ({len(assignments[c])} items): {c}")
                for i in assignments[c][:max_items_per_cluster]:
                    print(f"  {searches[i]}")
                print("\n")
            print("-" * 100)
            print("Residuals:")
            all_assignment_indices: set[int] = set()
            for c in assignments:
                all_assignment_indices.update(assignments[c])
            missing_indices = [i for i in range(len(searches)) if i not in all_assignment_indices]
            for i in missing_indices[:max_items_per_cluster]:
                print(f"  {searches[i]}")

        print("=" * 100)
        print(f"Positive Clusters for {len(self.search_results)} out of {self.total_search_results} searches:")
        _visualize(self.positive_clusters, self.search_results, self.positive_clusters_assignments)
        print("=" * 100)
        print(f"Negative Clusters for {len(self.near_search_results)} out of {self.total_neg_search_results} searches:")
        _visualize(self.negative_clusters, self.near_search_results, self.negative_clusters_assignments)
    
    def visualize_clusters_html(self):
        """Generate HTML visualization of clusters with collapsible sections."""
        def _visualize_html(clusters: list[str], searches: list[str], assignments: dict[str, list[int]]) -> str:
            html = ""
            for c in clusters:
                items = assignments[c]
                # Create collapsible section for each cluster
                html += f'''
                <details>
                    <summary><strong>Cluster ({len(items)} items):</strong> {c}</summary>
                    <div style="margin-left: 20px; margin-top: 10px;">
                        <ul>
                '''
                for i in items:
                    # HTML escape the search text
                    search_text = searches[i].replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
                    html += f'                            <li>{search_text}</li>\n'
                
                html += '''                        </ul>
                    </div>
                </details>
                '''
            
            # Add residuals section
            all_assignment_indices: set[int] = set()
            for c in assignments:
                all_assignment_indices.update(assignments[c])
            missing_indices = [i for i in range(len(searches)) if i not in all_assignment_indices]
            
            if missing_indices:
                html += f'''
                <details>
                    <summary><strong>Residuals ({len(missing_indices)} items):</strong> Unassigned items</summary>
                    <div style="margin-left: 20px; margin-top: 10px;">
                        <ul>
                '''
                for i in missing_indices:
                    search_text = searches[i].replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
                    html += f'                            <li>{search_text}</li>\n'
                
                html += '''                        </ul>
                    </div>
                </details>
                '''
            
            return html

        # Build complete HTML document
        html = f'''
    <h1>{self.search_query}</h1>
    
    <h2>Positive Clusters for {len(self.search_results)} out of {self.total_search_results} searches</h2>
'''
        
        html += _visualize_html(self.positive_clusters, self.search_results, self.positive_clusters_assignments)
        
        html += f'''
    <h2>Negative Clusters for {len(self.near_search_results)} out of {self.total_neg_search_results} searches</h2>
'''
        
        html += _visualize_html(self.negative_clusters, self.near_search_results, self.negative_clusters_assignments)
                
        return html

    @classmethod
    def finalize_html(cls, html: str):
        prefix = '''<!DOCTYPE html>
<html>
<head>
    <title>Cluster Visualization</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        h1 { color: #333; border-bottom: 2px solid #333; padding-bottom: 10px; }
        h2 { color: #666; border-bottom: 1px solid #666; padding-bottom: 5px; }
        details { margin-bottom: 15px; }
        summary { 
            cursor: pointer; 
            padding: 8px; 
            background-color: #f0f0f0; 
            border: 1px solid #ccc; 
            border-radius: 4px; 
            margin-bottom: 5px; 
        }
        summary:hover { background-color: #e0e0e0; }
        details[open] > summary { 
            border-bottom: none; 
            border-bottom-left-radius: 0; 
            border-bottom-right-radius: 0; 
        }
        details > div { 
            border: 1px solid #ccc; 
            border-top: none; 
            border-bottom-left-radius: 4px; 
            border-bottom-right-radius: 4px; 
            background-color: #fafafa; 
            padding: 10px; 
        }
        ul { margin: 0; padding-left: 20px; }
        li { margin-bottom: 5px; }
    </style>
</head>
<body>'''
        suffix = '''
</body>
</html>'''

        return prefix + '\n' + html + '\n' + suffix



In [ ]:
refiner = SearchRefiner(PICOCTF_TRANSCRIPTS, "environmental issues encountered by the agent")

await refiner.get_initial_search_results()
await refiner.get_initial_clusters()
await refiner.get_assign_residuals()
refiner.visualize_clusters()



In [ ]:
refiner2 = SearchRefiner(PICOCTF_TRANSCRIPTS, """environmental issues encountered by the agent.
this includes all kinds of missing tools and packages, output truncation, state persistence issues, network access issues, and inability to run or access binaries.
it does not include permissions issues, difficulties encoding and decoding characters, or timeouts from computational complexity""")

await refiner2.get_initial_search_results()
await refiner2.get_initial_clusters()
await refiner2.get_assign_residuals()
refiner2.visualize_clusters()


In [ ]:
await refiner2.visualize_search_results_for_cluster(
    """**State Persistence Issues in Python**: Variables, functions, and module imports defined in one Python tool execution weren't available in subsequent executions, requiring redefinition in each new code block, shown by numerous NameErrors for previously defined variables, functions, and missing imports.""",
    """State Persistence Issues: Variables, functions, and module imports defined in previous Python tool calls are not retained between executions, causing NameError exceptions when the agent attempts to reference them in subsequent calls.""",
    max_examples=250
)

In [ ]:
from docent_core._llm_util.providers.preferences import ModelOption


# await LlmApiClusterAssigner.from_o3_mini().


new_assigner = LlmApiClusterAssigner(
            system_prompt=None,
            max_new_tokens=8192,
            temperature=1,
            model_options=[
            ModelOption(
                provider="openai",
                model_name="o4-mini",
                reasoning_effort="medium",
            ),
        ]
        )


#11.8s, 64
#10.4s, 57
res = await new_assigner.assign(
    refiner.near_search_results, [
    """Declaring success prematurely without validation: The model confidently presented solutions as complete and successful despite lacking evidence or verification that the implementations actually resolved the issues correctly.""",
    ] * len(refiner.near_search_results)
)

print(sum(r[0] for r in res if r is not None))

In [ ]:
print(SearchRefiner.finalize_html(refiner.visualize_clusters_html() + refiner2.visualize_clusters_html()))

In [ ]:
refiner = SearchRefiner(BRIDGEWATER_TRANSCRIPTS, "obvious reasoning errors made by the forecaster")

await refiner.get_initial_search_results()
await refiner.get_initial_clusters()
await refiner.get_assign_residuals()
refiner.visualize_clusters()



In [ ]:
refiner2 = SearchRefiner(BRIDGEWATER_TRANSCRIPTS, """obvious reasoning errors made by the forecaster.
this includes not taking market efficiency into account, confusing short-term vs long-term horizons, overreliance on qualitative reasoning without backing from real data, neglecting base rates and reference classes, general overconfidence and incomplete calibration""")

await refiner2.get_initial_search_results()
await refiner2.get_initial_clusters()
await refiner2.get_assign_residuals()
refiner2.visualize_clusters()




In [ ]:
print(SearchRefiner.finalize_html(refiner.visualize_clusters_html() + refiner2.visualize_clusters_html()))

In [ ]:
refiner = SearchRefiner(ELICITATION_TRANSCRIPTS, "investigator model outputs a prompt that violate the instructions that were provided")

await refiner.get_initial_search_results()
await refiner.get_initial_clusters()
await refiner.get_assign_residuals()
refiner.visualize_clusters()



In [ ]:
refiner2 = SearchRefiner(ELICITATION_TRANSCRIPTS, """investigator model outputs a prompt that violate the instructions that were provided.
this includes adding meta-discussion about the prompt / its goals, providing instructions about model behavior, and engaging in hypotheticals.
it does not include using quotation marks or requesting private information
""")

await refiner2.get_initial_search_results()
await refiner2.get_initial_clusters()
await refiner2.get_assign_residuals()
refiner2.visualize_clusters()



In [ ]:
print(SearchRefiner.finalize_html(refiner.visualize_clusters_html() + refiner2.visualize_clusters_html()))

In [ ]:
refiner = SearchRefiner(CURSOR_TRANSCRIPTS, "times the model used an unintended shortcut to satisfy the request")

await refiner.get_initial_search_results()
await refiner.get_initial_clusters()
await refiner.get_assign_residuals()
refiner.visualize_clusters()



In [ ]:
# await refiner.visualize_search_results_for_cluster(
#     "",
#     """Premature Implementation: Making significant code changes or implementing new features without first understanding requirements, checking existing implementations, or confirming with the user.""",
#     max_examples=250
# )

In [ ]:
refiner2 = SearchRefiner(CURSOR_TRANSCRIPTS, """times the model used an unintended shortcut to satisfy the request.
this includes skipping environment setup issues, implementing code changes prematurely, and avoiding proper testing or error handling.
it does not include enhancing existing implementations instead of creating new ones
""")

await refiner2.get_initial_search_results()
await refiner2.get_initial_clusters()
await refiner2.get_assign_residuals()
refiner2.visualize_clusters()



In [ ]:
await refiner2.visualize_search_results_for_cluster(
    """Avoiding proper testing: Instances where the model implemented changes without creating or suggesting appropriate testing strategies, verification procedures, or validation methodologies to confirm the solution worked correctly.""",
    """Implementing solutions without proper testing or verification: The model made code changes without suggesting, implementing, or performing adequate testing to verify the effectiveness of the changes, including not confirming mathematical correctness of algorithms or validating fixes.""",
    max_examples=250
)

In [ ]:
print(SearchRefiner.finalize_html(refiner.visualize_clusters_html() + refiner2.visualize_clusters_html()))